Bronze data processing

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/akhildataengineer@hotmail.com/Atlikon_migration_pipeline/1_setup/3_utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-store-dp1/{data_source}/' #s3://sportsbar-store-dp/customers/*.csv
landing_path = f'{base_path}/landing/'
processing_path = f'{base_path}/processed/'


In [0]:
dbutils.fs.ls('s3://sportsbar-store-dp1/')

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df = spark.read\
            .options(header=True, inferSchema=True)\
            .csv(f"{landing_path}/*.csv")\
            .withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

In [0]:
df.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)

Silver Data Processing

In [0]:
# Moving files to processed folder
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processing_path}/{file_info.name}",
        True
    )

In [0]:
df_enr = spark.sql(f"SELECT * FROM {bronze_table}")
df_enr.show(2)

In [0]:
df_enr.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_enr.columns]).show()

In [0]:
# Remove all nulls
df_enr = df_enr.dropna(how='any')
df_enr.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_enr.columns]).show()

In [0]:
display(df_enr.select("order_placement_date").distinct())

In [0]:
# Clean Ids
df_enr = df_enr.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

In [0]:
# Normalize dates
df_enr = df_enr.withColumn("order_placement_date", F.regexp_replace(F.col("order_placement_date"), r"^[a-zA-Z]+,\s", ""))

df_enr = df_enr.withColumn("order_placement_date", F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
))

In [0]:
# Drop Duplicates
print(f"count before: {df_enr.count()}")
df_enr = df_enr.dropDuplicates()
print(f"count after: {df_enr.count()}")

In [0]:
# Cast product_id as string
df_enr = df_enr.withColumn("product_id", F.col("product_id").cast("string"))

In [0]:
# join products
df_prod = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.products")
df_joined = df_enr.join(df_prod, on = "product_id", how = "inner").select(df_enr["*"], df_prod["product_code"])
df_joined.show()

In [0]:
# Create table if not exist or update table
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write\
             .format("delta")\
             .option("delta.enableChangeDataFeed", "true")\
             .option("mergeSchema", "true")\
             .mode("overwrite")\
             .saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver")\
                .merge(
                    df_joined.alias("bronze"), 
                    "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id")\
                .whenMatchedUpdateAll()\
                .whenNotMatchedInsertAll()\
                .execute()

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write\
            .format("delta")\
            .option("delta.enableChangeDataFeed", "true")\
            .option("mergeSchema", "true")\
            .mode("overwrite")\
            .saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source")\
            .merge(df_gold.alias("gold"), 
                "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()

Merge with Parent table

In [0]:
df_children_fact = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_children_fact.show(10)

In [0]:
df_monthly = (
    df_children_fact
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_{data_source}")
gold_parent_delta.alias("parent_gold")\
        .merge(df_monthly.alias("child_gold"), 
            "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()